# product-slm: QLoRA fine-tune on a free T4 (Colab or Kaggle)

The notebook runs these steps:
1. Install the repo with the `train` extra and `bitsandbytes`.
2. Run QLoRA on **Qwen2.5-1.5B-Instruct** over the 600 distilled train labels. This takes ~10–20 min on a T4.
3. Evaluate on val and test with the same runner and metrics as every other model in the repo. It also runs the **zero-shot base model** with the same short prompt, so the fine-tuning gain is measured, not assumed.
4. Package everything for download:
   - the adapter;
   - the predictions;
   - the cassettes, which let CI replay these exact outputs without a GPU;
   - the training summary.

**Runtime → Change runtime type → T4 GPU** before running anything. Then run all cells: `sh()` raises on the first failing command, so a broken step stops the notebook right where it failed.

In [ ]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    """Run a shell command with live output; raise if it fails (a plain `!cmd` would carry on silently)."""
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {cmd}")


print(sys.version)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
)
if gpu.returncode != 0:
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun.")
print(gpu.stdout)

In [ ]:
REPO = "/content/product-slm"
if os.path.isdir(REPO):
    sh(f"git -C {REPO} pull -q")  # rerunning the notebook picks up fixes pushed since the first run
else:
    sh(f"git clone -q https://github.com/houssem019/product-slm.git {REPO}")
os.chdir(REPO)
# Colab ships CUDA builds of torch/transformers/accelerate; the `train` extra's floors are low enough that pip keeps
# them instead of downloading a multi-GB torch. bitsandbytes is the one extra (4-bit kernels, CUDA only).
sh('pip install -q -e ".[train]" bitsandbytes')
sh(
    'python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, torch.cuda.is_available())"'
)

## 1. Build the SFT data
The SFT data is chat triples (short student prompt, title, canonical JSON). The test split is never exported for training.

In [ ]:
sh("python -m pslm.train.format")

## 2. QLoRA training
These settings come from `pslm/train/train.py`:
- **Base model:** 4-bit NF4 with double quantization, fp16 compute (a T4 has no bf16).
- **LoRA:** r=16, alpha=32 on all attention and MLP projections.
- **Schedule:** 3 epochs, lr 2e-4 with cosine decay, effective batch 16, gradient checkpointing.
- **Loss:** answer tokens only.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "outputs/qwen2.5-1.5b-lora"
sh(f"python -m pslm.train.train --base {BASE} --load-in-4bit --out {OUT}")

In [ ]:
import json
from pathlib import Path

s = json.loads(Path(f"{OUT}/train_summary.json").read_text())
print(
    {k: s[k] for k in ("train_loss", "eval_loss", "runtime_s", "device", "trainable_params", "total_params")}
)
for e in s["log_history"]:
    if "eval_loss" in e:
        print(f"epoch {e['epoch']:.0f}: eval_loss {e['eval_loss']:.4f}")

## 3. Evaluate: fine-tuned vs zero-shot base, on val and test
- **Decoding:** greedy and unconstrained, so JSON validity is the model's own. vLLM guided decoding comes at serving time.
- **`--record`:** writes cassettes, so CI can replay these runs on a CPU-only runner.

In [ ]:
for split in ("val", "test"):
    sh(
        f"python -m pslm.distill.label {split} --provider hf --model {OUT}/adapter --name qwen2.5-1.5b-lora --record"
    )
    sh(
        f"python -m pslm.distill.label {split} --provider hf --model {BASE} --name qwen2.5-1.5b-zeroshot"
        " --prompt student --record"
    )

## 4. Download the artifacts
Unzip the archive at the repo root locally. `outputs/` stays gitignored; the adapter goes to the Hub, or stays local.

In [ ]:
sh(f"zip -qr slm_results.zip data/preds data/cassettes {OUT}/train_summary.json {OUT}/adapter")
try:
    from google.colab import files

    files.download("slm_results.zip")
except ImportError:  # Kaggle: download it from the Output tab
    print("slm_results.zip written to", os.getcwd())

Optional: publish the adapter on the Hugging Face Hub.
1. Run `from huggingface_hub import login; login()` with a write token.
2. Add `--push-to-hub houssem019/product-slm-qwen2.5-1.5b-lora` to the training command in step 2.